## Created from MODIS and VIIRS active fires for 2023 and 2024.
We downloaded the af files from NASA (https://firms.modaps.eosdis.nasa.gov/data/download/DL_FIRE_SV-C2_578623.zip) and the burned areas from effis. 
Then we:
 - Created a buffer of 1000 metres in the burned area polygons
 - Filtered to take only the summer months (May - September)
 - Selected only the af that intersect with the buffered fire polygons and deleted the rest
 - Performed spatial join between af and bsm polygons and compared their dates (if their distance in days was more than 4 days we deleted the af)

In [1]:
import xarray as xr
import autoroot
from satpy.scene import Scene
from pyhdf.SD import SD, SDC 
import pandas as pd
import cartopy.crs as ccrs
import rioxarray
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime
import fnmatch
from pyproj import CRS, Transformer
from pyhdf.SD import SD, SDC 
from rs_tools._src.geoprocessing.match import match_timestamps_af
from rs_tools._src.utils.io import get_list_filenames
from pathlib import Path
from tqdm import tqdm


/home/sgirtsou/miniconda3/envs/rs_tools/lib/python3.11/site-packages/pyproj/__init__.py:95: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()
/home/sgirtsou/miniconda3/envs/rs_tools/lib/python3.11/site-packages/goes2go/data.py:665: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  within=pd.to_timedelta(config["nearesttime"].get("within", "1h")),
/home/sgirtsou/miniconda3/envs/rs_tools/lib/python3.11/site-packages/goes2go/NEW.py:188: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  within=pd.to_timedelta(config["nearesttime"].get("within", "1h")),


In [19]:
year = 2023

In [20]:
def convert_lat_lon_to_x_y(crs, lon, lat):
    transformer = Transformer.from_crs(CRS("+proj=latlon"), crs, always_xy=True)
    x, y = transformer.transform(lon, lat)
    return x, y
def parse_af_dates_from_file(file):
    timestamp = Path(file).name.split("_")[0]
    return timestamp

In [21]:
msg_path = '/mnt/outputs/geoprocessed'
save_af_path = f'/mnt/data8tb/fire_detection/datasets/pointcloud_5/geoprocessed_af'
os.makedirs(save_af_path, exist_ok=True)

In [22]:
af = pd.read_csv(f'/home/sgirtsou/Projects/rs_tools/data/fire-detection/nasa_af_point_cloud_5_{str(year)}.csv')

In [23]:
af.ACQ_DATE

0        2023/07/15
1        2023/07/15
2        2023/07/15
3        2023/07/15
4        2023/07/15
            ...    
34660    2023/05/19
34661    2023/05/19
34662    2023/05/19
34663    2023/05/19
34664    2023/05/19
Name: ACQ_DATE, Length: 34665, dtype: object

In [24]:
af['ACQ_DATE'] = af['ACQ_DATE'].str.replace('/', '-')
af['ACQ_DATETIME'] = pd.to_datetime(af['ACQ_DATE'] + ' ' + af['ACQ_TIME'].astype(str).str.zfill(4), format='%Y-%m-%d %H%M')
af['datetime'] = af['ACQ_DATETIME'].dt.strftime('%Y%m%d%H%M00')

In [25]:
af_sel = af[af['ACQ_DATETIME'].dt.year == year]

In [26]:
af.shape, af_sel.shape

((34665, 16), (34665, 16))

In [27]:
year

2023

In [28]:
af_sel.drop_duplicates(subset=['LATITUDE','LONGITUDE','ACQ_DATE', 'ACQ_TIME'], inplace=True)

In [29]:
unique_times_af = af_sel['ACQ_DATETIME'].dt.strftime('%Y%m%d%H%M00').unique().tolist()

In [30]:
unique_times_af

['20230715110800',
 '20230909003700',
 '20230715101900',
 '20230909012800',
 '20230918010900',
 '20230828122500',
 '20230725014100',
 '20230828113600',
 '20230725005100',
 '20230922125500',
 '20230819124400',
 '20230922120600',
 '20230821004400',
 '20230823024100',
 '20230824022100',
 '20230717121200',
 '20230726110400',
 '20230823015000',
 '20230823130900',
 '20230824013100',
 '20230720004300',
 '20230725112100',
 '20230725121300',
 '20230819115300',
 '20230823121900',
 '20230823135900',
 '20230725130400',
 '20230724114200',
 '20230724123200',
 '20230705124000',
 '20230822123800',
 '20230822114700',
 '20230806123800',
 '20230806114800',
 '20230822002600',
 '20230823005600',
 '20230722112900',
 '20230823000600',
 '20230823014700',
 '20230812104300',
 '20230922111700',
 '20230812113500',
 '20230722121700',
 '20230928101100',
 '20230719235400',
 '20230720004500',
 '20230721002600',
 '20230709121300',
 '20230724213800',
 '20230725005200',
 '20230821013400',
 '20230821004300',
 '2023072413

In [31]:
files_msg_af = get_list_filenames(msg_path, ".nc", str(year))
unique_times_msg = list(set(map(parse_af_dates_from_file, files_msg_af)))
df_matches = match_timestamps_af(unique_times_af, unique_times_msg, cutoff=15)
df_matches.columns = ['timestamp_af', 'timestamp_msg']

In [32]:
df_matches

,timestamp_af,timestamp_msg
0,20230715110800,20230715111241
1,20230909003700,20230909004242
2,20230715101900,20230715102742
3,20230909012800,20230909014242
4,20230918010900,20230918011242
...,...,...
689,20230725115200,20230725115742
690,20230725190200,20230725191241
691,20230725204000,20230725204241
692,20230726084100,20230726084242


In [33]:
def create_fires_ds(df_af_datetime, msg):
    array = np.zeros((msg.y.size, msg.x.size))
    var = "msg_seviri_fes_3km"
    crs_wkt = msg[var].crs_wkt
    crs = CRS(crs_wkt)
    for index, row in df_af_datetime.iterrows():
        lon = row['LONGITUDE']
        lat = row['LATITUDE']
        x_sel, y_sel = convert_lat_lon_to_x_y(crs, lon, lat)
        selected = msg.sel(x=x_sel, y=y_sel, method='nearest')
        # Get the indices of the nearest point
        x_idx = msg.get_index('x').get_loc(selected['x'].item())
        y_idx = msg.get_index('y').get_loc(selected['y'].item())
        try:
            array[y_idx, x_idx] = 1
        except IndexError:
            print(f"Index out of bounds: y_idx={y_idx}, x_idx={x_idx}, array shape={array.shape}")
    da = xr.DataArray(
        array,
        coords={"y": msg.y, "x": msg.x},
        dims=("y", "x")
    )
    da.attrs['af_time'] = df_af_datetime.datetime.unique()[0]
    da.attrs['DAYNIGHT'] = df_af_datetime.DAYNIGHT.unique()[0]
    return da

In [34]:
for index, row in tqdm(df_matches.iterrows(), total=len(df_matches), desc="Processing files"):
    df_af_datetime = af_sel[af_sel.datetime == row['timestamp_af']]
    msg = xr.open_dataset(os.path.join(msg_path, f"{row.timestamp_msg}_msg.nc"))
    fires_ds = create_fires_ds(df_af_datetime, msg)
    fires_ds.to_netcdf(os.path.join(save_af_path, f"{row.timestamp_msg}_af.nc"))

Processing files:   0%|          | 0/694 [00:00<?, ?it/s]

Processing files: 100%|██████████| 694/694 [11:44<00:00,  1.02s/it]  


## After this I run the prepatcher for af

### prepatcher_af.py from VScode (not terminal)
    prepatch(read_path = '/mnt/data8tb/fire_detection/af_nasa_geoprocessed/2023/', save_path='/mnt/data8tb/fire_detection/af_nasa_patched/2023/', patch_size=32, stride_size=32, fire_cutoff=1, save_filetype='tif')


## Copy msg patches to have harmonized folders
### I did the copy from /mnt/data8tb/fire_detection/copy_msg_geoprocessed.sh

In [ ]:
missing = []

In [ ]:
import os
import shutil

# Define source and destination directories
af_dir = save_af_path
msg_dir = msg_path
dest_dir = "/mnt/data8tb/fire_detection/msg_geoprocessed/2023"

# Get the list of _af.nc files
af_files = [f for f in os.listdir(af_dir) if f.endswith("nc")]

# Extract timestamps and copy corresponding _msg.nc files
for af_file in af_files:
    timestamp = af_file.split("_")[0]  # Extract timestamp (first part of the filename)
    msg_file = f"{timestamp}_msg.nc"
    if os.path.exists(os.path.join(dest_dir, msg_file)):  # Check if the _msg.nc file exists
        continue
    msg_path = os.path.join(msg_dir, msg_file)
    try:
        shutil.copy(msg_path, os.path.join(dest_dir, msg_file))  # Copy the file
        print(f"Copied: {msg_file}")
        time.sleep(0.1)
    except:
        missing.append(msg_file)
        print(f"Missing: {msg_file}")

Copied: 20230904095742_msg.nc
Missing: 20230904095742_msg.nc
Copied: 20230908121241_msg.nc
Missing: 20230908121241_msg.nc
Copied: 20230625104241_msg.nc
Missing: 20230625104241_msg.nc
Copied: 20230730001241_msg.nc
Missing: 20230730001241_msg.nc
Copied: 20230505142741_msg.nc
Missing: 20230505142741_msg.nc
Copied: 20230830124241_msg.nc
Missing: 20230830124241_msg.nc
Copied: 20230721212741_msg.nc
Missing: 20230721212741_msg.nc
Copied: 20230713012742_msg.nc
Missing: 20230713012742_msg.nc
Copied: 20230826024241_msg.nc
Missing: 20230826024241_msg.nc
Copied: 20230810024242_msg.nc
Missing: 20230810024242_msg.nc
Copied: 20230830094242_msg.nc
Missing: 20230830094242_msg.nc
Copied: 20230710232742_msg.nc
Missing: 20230710232742_msg.nc


## Run patching for msg files

## Delete the patches that are not common to af patches 
### af patches where created only when a fire pixel was detected inside the 32x32 pixel. for this reason the msg patches will be much more. 

In [75]:
import os
import shutil

af_patches = "/mnt/data8tb/fire_detection/datasets/pointcloud_5/masks/"
msg_patches = "/mnt/data8tb/fire_detection/datasets/pointcloud_5/images"

In [76]:
for file in os.listdir(msg_patches):
    if os.path.exists(os.path.join(af_patches, file)):
        continue
    else:
        os.remove(os.path.join(msg_patches, file))

In [77]:
for file in os.listdir(af_patches):
    if os.path.exists(os.path.join(msg_patches, file)):
        continue
    else:
        print(f"No file found for {file}")
        #os.remove(os.path.join(af_patches, file))

## This was created to just copy the msg files according to pointcloud_5 to avoid copying again the geoprocessed from /mnt/outputs.
### In theory all the pointcloud_5 patches will be present in the msg patces created without cutting off the number of af points in a burned area
#### It did not work :( ->> recopy msg geoprocessed and rerunning msg_patching

In [71]:
import os
import shutil

af_patches = "/mnt/data8tb/fire_detection/datasets/pointcloud_5/masks/"
msg_patches = "/mnt/outputs/patched"
msg_patches_destination = "/mnt/data8tb/fire_detection/datasets/pointcloud_5/images/"

In [54]:
missing_files = []

In [55]:
for file in os.listdir(af_patches):
    if os.path.exists(os.path.join(msg_patches, file)):
        shutil.copy(os.path.join(msg_patches, file), os.path.join(msg_patches_destination, file))
    else:
        print(f"No file found for {file}")
        missing_files.append(file)
        #os.remove(os.path.join(af_patches, file))

No file found for 20230808022742_patch_229.tif
No file found for 20230717234242_patch_258.tif
No file found for 20230725225741_patch_185.tif
No file found for 20230813112742_patch_212.tif
No file found for 20230715004242_patch_221.tif
No file found for 20230724101242_patch_185.tif
No file found for 20230721232741_patch_185.tif
No file found for 20230719231241_patch_185.tif
No file found for 20230722114242_patch_185.tif
No file found for 20230724231241_patch_258.tif
No file found for 20230723011241_patch_185.tif
No file found for 20230718125742_patch_211.tif
No file found for 20230707121242_patch_202.tif
No file found for 20230714235741_patch_221.tif
No file found for 20230725005741_patch_366.tif
No file found for 20230726012741_patch_396.tif
No file found for 20230725112742_patch_185.tif
No file found for 20230806021241_patch_229.tif
No file found for 20230922125741_patch_248.tif
No file found for 20230706011242_patch_211.tif
No file found for 20230808014242_patch_229.tif
No file found

In [58]:
missing_files

['20230808022742_patch_229.tif',
 '20230717234242_patch_258.tif',
 '20230725225741_patch_185.tif',
 '20230813112742_patch_212.tif',
 '20230715004242_patch_221.tif',
 '20230724101242_patch_185.tif',
 '20230721232741_patch_185.tif',
 '20230719231241_patch_185.tif',
 '20230722114242_patch_185.tif',
 '20230724231241_patch_258.tif',
 '20230723011241_patch_185.tif',
 '20230718125742_patch_211.tif',
 '20230707121242_patch_202.tif',
 '20230714235741_patch_221.tif',
 '20230725005741_patch_366.tif',
 '20230726012741_patch_396.tif',
 '20230725112742_patch_185.tif',
 '20230806021241_patch_229.tif',
 '20230922125741_patch_248.tif',
 '20230706011242_patch_211.tif',
 '20230808014242_patch_229.tif',
 '20230918021243_patch_204.tif',
 '20230828014242_patch_248.tif',
 '20230827022742_patch_203.tif',
 '20230725005741_patch_248.tif',
 '20230725005741_patch_185.tif',
 '20230726105741_patch_185.tif',
 '20230722235742_patch_185.tif',
 '20230725014242_patch_366.tif',
 '20230706001242_patch_211.tif',
 '20230724

In [57]:
len(os.listdir(af_patches))

777

In [59]:
import rioxarray

In [69]:
files_dict = {}
for file in missing_files:
    ds = rioxarray.open_rasterio(os.path.join(af_patches, file))
    files_dict[file] = ds.squeeze().values.sum()

In [70]:
files_dict

{'20230808022742_patch_229.tif': 1.0,
 '20230717234242_patch_258.tif': 5.0,
 '20230725225741_patch_185.tif': 7.0,
 '20230813112742_patch_212.tif': 1.0,
 '20230715004242_patch_221.tif': 4.0,
 '20230724101242_patch_185.tif': 8.0,
 '20230721232741_patch_185.tif': 6.0,
 '20230719231241_patch_185.tif': 4.0,
 '20230722114242_patch_185.tif': 12.0,
 '20230724231241_patch_258.tif': 3.0,
 '20230723011241_patch_185.tif': 9.0,
 '20230718125742_patch_211.tif': 2.0,
 '20230707121242_patch_202.tif': 1.0,
 '20230714235741_patch_221.tif': 4.0,
 '20230725005741_patch_366.tif': 2.0,
 '20230726012741_patch_396.tif': 2.0,
 '20230725112742_patch_185.tif': 1.0,
 '20230806021241_patch_229.tif': 1.0,
 '20230922125741_patch_248.tif': 4.0,
 '20230706011242_patch_211.tif': 2.0,
 '20230808014242_patch_229.tif': 1.0,
 '20230918021243_patch_204.tif': 2.0,
 '20230828014242_patch_248.tif': 4.0,
 '20230827022742_patch_203.tif': 2.0,
 '20230725005741_patch_248.tif': 8.0,
 '20230725005741_patch_185.tif': 1.0,
 '202307261

In [68]:
ds.squeeze().values.sum()

6.0

In [72]:
af_geop_path = '/mnt/data8tb/fire_detection/datasets/pointcloud_5/geoprocessed_af'
msg_geop_path = '/mnt/data8tb/fire_detection/datasets/pointcloud_5/geoprocessed_msg'
af_files = os.listdir(af_geop_path)
msg_files = os.listdir(msg_geop_path)

In [74]:
len(af_files), len(msg_files)

(536, 536)